# Motion Feature Extraction - Shape Tracking

이 노트북은 motion feature 추출 과정에서의 **shape만 추적**합니다.
- Optical flow 계산 제외
- cumsum, min, max 등의 실제 계산 제외
- frame_count, total_steps, 배열 길이 등 dimension만 추적

목적: dimension mismatch 디버깅 및 예상 shape 확인

In [1]:
import sys
sys.path.insert(0, '/tf/01_code/mylittlecodes/SleepVST_baseline')

from src.data.preprocess.motion_shape_tracker import (
    track_video_shapes,
    batch_track_shapes,
    compare_shapes_with_actual,
    print_comparison,
    ShapeTrackingResult
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

## 1. 단일 Record의 Shape 추적

특정 record의 모든 dimension을 추적합니다.

In [2]:
# 추적할 record ID
record_id = 'A2020-EM-01-0184'

# Shape 추적 (계산 없이 dimension만)
result = track_video_shapes(
    record_id=record_id,
    target_fps=4,
    verbose=True
)

Shape Tracking Result for A2020-EM-01-0184

VIDEO PROPERTIES:
  Path: /tf/00_data/#_2021_Sleep_Video/A2020-EM-01-0184/A2020-EM-01-0184_video_01.mp4
  Exists: True
  FPS: 4.93
  Frame Count: 141,853
  Frame Size: 640 x 480
  Duration: 28770.19 seconds (479.50 minutes)

PROCESSING PARAMETERS:
  Target FPS: 4
  Frame Interval: 1 (every 1th frame)

CALCULATED DIMENSIONS:
  Total Steps: 141,852
  Frames to Process: 141,852
  Expected Sequence Length: 141,852

EXPECTED OUTPUT:
  Expected Epochs (30s): 959.01
  Feature Length / Expected Epochs: 147.915624

FEATURE SHAPES:
  f1@300s_Body                   -> (141852,)
  f1@300s_Head                   -> (141852,)
  f1@300s_Outer                  -> (141852,)
  f1@30s_Body                    -> (141852,)
  f1@30s_Head                    -> (141852,)
  f1@30s_Outer                   -> (141852,)
  f2@300s_Body                   -> (141852,)
  f2@300s_Head                   -> (141852,)
  f2@300s_Outer                  -> (141852,)
  f2@30s_Body 

## 2. Shape 추적 상세 분석

각 단계별로 어떻게 shape가 변하는지 확인합니다.

In [3]:
if result and result.video_exists:
    print("\n" + "="*80)
    print("STEP-BY-STEP SHAPE CALCULATION")
    print("="*80)
    
    print(f"\nStep 1: Video Properties")
    print(f"  - Total frames in video: {result.frame_count:,}")
    print(f"  - Original FPS: {result.fps:.2f}")
    print(f"  - Video duration: {result.duration_seconds:.2f} seconds")
    
    print(f"\nStep 2: Frame Interval Calculation")
    print(f"  - Target FPS: {result.target_fps}")
    print(f"  - Frame interval = round(fps / target_fps) = round({result.fps:.2f} / {result.target_fps}) = {result.frame_interval}")
    print(f"  - This means we take every {result.frame_interval}th frame")
    
    print(f"\nStep 3: Total Steps Calculation")
    print(f"  - Formula: total_steps = max(1, int(frame_count // frame_interval) - 1)")
    print(f"  - total_steps = max(1, int({result.frame_count} // {result.frame_interval}) - 1)")
    print(f"  - total_steps = max(1, int({result.frame_count // result.frame_interval}) - 1)")
    print(f"  - total_steps = max(1, {result.frame_count // result.frame_interval - 1})")
    print(f"  - total_steps = {result.total_steps:,}")
    
    print(f"\nStep 4: Frame Processing Loop")
    print(f"  - Loop runs for 'total_steps' iterations: {result.total_steps:,} times")
    print(f"  - Each iteration:")
    print(f"    1. Skip {result.frame_interval - 1} frames")
    print(f"    2. Read 1 frame")
    print(f"    3. Compute optical flow (would happen here, but we skip for shape tracking)")
    print(f"    4. Extract region magnitudes (would happen here)")
    print(f"    5. Append to region_seqs['Head']['v'], region_seqs['Head']['s'], etc.")
    print(f"  - Frames processed: {result.frames_that_would_be_processed:,}")
    
    print(f"\nStep 5: Region Sequences Shape")
    print(f"  - After the loop, region_seqs would have:")
    print(f"    * region_seqs['Head']['v']: list of length {result.expected_sequence_length:,}")
    print(f"    * region_seqs['Head']['s']: list of length {result.expected_sequence_length:,}")
    print(f"    * region_seqs['Body']['v']: list of length {result.expected_sequence_length:,}")
    print(f"    * region_seqs['Body']['s']: list of length {result.expected_sequence_length:,}")
    print(f"    * region_seqs['Outer']['v']: list of length {result.expected_sequence_length:,}")
    print(f"    * region_seqs['Outer']['s']: list of length {result.expected_sequence_length:,}")
    
    print(f"\nStep 6: Convert to NumPy Arrays")
    print(f"  - v_seq = np.array(region_seqs['Head']['v'])  -> shape: ({result.expected_sequence_length},)")
    print(f"  - s_seq = np.array(region_seqs['Head']['s'])  -> shape: ({result.expected_sequence_length},)")
    
    print(f"\nStep 7: Compute Motion Features")
    print(f"  - Input: v_seq and s_seq, both shape ({result.expected_sequence_length},)")
    print(f"  - All output features will have shape: ({result.expected_sequence_length},)")
    print(f"  - Total features: {len(result.feature_shapes)}")
    
    print(f"\nStep 8: Expected vs Actual Epochs")
    print(f"  - Expected epochs (30s): {result.expected_epochs:.2f}")
    print(f"  - Feature length: {result.expected_sequence_length:,}")
    print(f"  - Ratio (feature_length / expected_epochs): {result.expected_sequence_length / result.expected_epochs:.6f}")
    print(f"  - This ratio should be close to target_fps * 30 = {result.target_fps * 30} if perfect")
    
    print("\n" + "="*80)


STEP-BY-STEP SHAPE CALCULATION

Step 1: Video Properties
  - Total frames in video: 141,853
  - Original FPS: 4.93
  - Video duration: 28770.19 seconds

Step 2: Frame Interval Calculation
  - Target FPS: 4
  - Frame interval = round(fps / target_fps) = round(4.93 / 4) = 1
  - This means we take every 1th frame

Step 3: Total Steps Calculation
  - Formula: total_steps = max(1, int(frame_count // frame_interval) - 1)
  - total_steps = max(1, int(141853 // 1) - 1)
  - total_steps = max(1, int(141853) - 1)
  - total_steps = max(1, 141852)
  - total_steps = 141,852

Step 4: Frame Processing Loop
  - Loop runs for 'total_steps' iterations: 141,852 times
  - Each iteration:
    1. Skip 0 frames
    2. Read 1 frame
    3. Compute optical flow (would happen here, but we skip for shape tracking)
    4. Extract region magnitudes (would happen here)
    5. Append to region_seqs['Head']['v'], region_seqs['Head']['s'], etc.
  - Frames processed: 141,852

Step 5: Region Sequences Shape
  - After the

## 3. 여러 Record의 Shape 비교

여러 record의 shape를 일괄적으로 추적하고 비교합니다.

In [ ]:
# 추적할 record IDs
test_records = ['A-0038', 'A-0039', 'A-0040', 'A-0041', 'A-0042']

# Batch shape tracking
results = batch_track_shapes(
    record_ids=test_records,
    target_fps=4,
    verbose=False  # 개별 출력 안함
)

# 결과를 DataFrame으로 정리
data = []
for record_id, result in results.items():
    if result.video_exists:
        data.append({
            'Record ID': record_id,
            'FPS': result.fps,
            'Frame Count': result.frame_count,
            'Duration (min)': result.duration_seconds / 60,
            'Frame Interval': result.frame_interval,
            'Total Steps': result.total_steps,
            'Sequence Length': result.expected_sequence_length,
            'Expected Epochs': result.expected_epochs,
            'Length/Epochs': result.expected_sequence_length / result.expected_epochs if result.expected_epochs > 0 else 0,
        })

df = pd.DataFrame(data)
print("\nShape Tracking Summary:")
print(df.to_string(index=False))

## 4. 계산 과정 시뮬레이션

실제 계산 없이 shape 변화만 시뮬레이션합니다.

In [5]:
def simulate_processing_steps(record_id='A-0038'):
    """실제 계산 없이 processing 과정의 shape만 추적"""
    
    result = track_video_shapes(record_id, verbose=False)
    
    if not result.video_exists:
        print(f"Video not found for {record_id}")
        return
    
    print(f"\n{'='*80}")
    print(f"Processing Simulation for {record_id}")
    print(f"{'='*80}\n")
    
    # Simulate the processing loop
    print(f"Starting processing loop...\n")
    
    # Initialize empty lists (simulated)
    v_head, s_head = [], []
    v_body, s_body = [], []
    v_outer, s_outer = [], []
    
    print(f"Initial state:")
    print(f"  v_head: length = 0")
    print(f"  s_head: length = 0")
    
    # Simulate loop
    for step in range(result.total_steps):
        # Each iteration appends one value
        v_head.append(0.0)  # Would be actual value
        s_head.append(0.0)  # Would be actual value
        v_body.append(0.0)
        s_body.append(0.0)
        v_outer.append(0.0)
        s_outer.append(0.0)
        
        # Print progress at certain intervals
        if step in [0, 10, 100, 1000] or step == result.total_steps - 1:
            print(f"\nAfter step {step + 1}:")
            print(f"  v_head: length = {len(v_head)}")
            print(f"  s_head: length = {len(s_head)}")
    
    print(f"\n\nLoop completed. Final lengths:")
    print(f"  v_head: {len(v_head)}")
    print(f"  s_head: {len(s_head)}")
    print(f"  v_body: {len(v_body)}")
    print(f"  s_body: {len(s_body)}")
    print(f"  v_outer: {len(v_outer)}")
    print(f"  s_outer: {len(s_outer)}")
    
    # Convert to numpy arrays (simulated)
    print(f"\nConverting to NumPy arrays...")
    v_seq = np.zeros(len(v_head))  # Would be np.array(v_head)
    s_seq = np.zeros(len(s_head))  # Would be np.array(s_head)
    
    print(f"  v_seq.shape: {v_seq.shape}")
    print(f"  s_seq.shape: {s_seq.shape}")
    
    # Feature computation (simulated)
    print(f"\nComputing features...")
    T = len(v_seq)
    
    # Time-based features
    for sec in [30, 300]:
        f1 = np.zeros(T)  # Would be actual cumsum computation
        f2 = np.zeros(T)
        print(f"  f1@{sec}s_Head.shape: {f1.shape}")
        print(f"  f2@{sec}s_Head.shape: {f2.shape}")
    
    # Threshold features
    for threshold in [0.01, 0.1, 1.0]:
        f3 = np.zeros(T, dtype=np.int32)
        f4 = np.zeros(T, dtype=np.int32)
        print(f"  f3@{threshold}_Head.shape: {f3.shape}")
        print(f"  f4@{threshold}_Head.shape: {f4.shape}")
    
    print(f"\nAll features have shape: ({T},)")
    print(f"Total features per region: 10")
    print(f"Total regions: 3 (Head, Body, Outer)")
    print(f"Total features: 30")
    
    print(f"\n{'='*80}\n")

# Run simulation
simulate_processing_steps('A2020-EM-01-0184')


Processing Simulation for A2020-EM-01-0184

Starting processing loop...

Initial state:
  v_head: length = 0
  s_head: length = 0

After step 1:
  v_head: length = 1
  s_head: length = 1

After step 11:
  v_head: length = 11
  s_head: length = 11

After step 101:
  v_head: length = 101
  s_head: length = 101

After step 1001:
  v_head: length = 1001
  s_head: length = 1001

After step 141852:
  v_head: length = 141852
  s_head: length = 141852


Loop completed. Final lengths:
  v_head: 141852
  s_head: 141852
  v_body: 141852
  s_body: 141852
  v_outer: 141852
  s_outer: 141852

Converting to NumPy arrays...
  v_seq.shape: (141852,)
  s_seq.shape: (141852,)

Computing features...
  f1@30s_Head.shape: (141852,)
  f2@30s_Head.shape: (141852,)
  f1@300s_Head.shape: (141852,)
  f2@300s_Head.shape: (141852,)
  f3@0.01_Head.shape: (141852,)
  f4@0.01_Head.shape: (141852,)
  f3@0.1_Head.shape: (141852,)
  f4@0.1_Head.shape: (141852,)
  f3@1.0_Head.shape: (141852,)
  f4@1.0_Head.shape: (14185

## 5. 실제 추출된 Feature와 비교

만약 이미 추출된 motion feature가 있다면, 예상한 shape와 비교합니다.

In [6]:
# 특정 record의 shape 비교
record_id = 'A2020-EM-01-0184'

comparison = compare_shapes_with_actual(
    record_id=record_id,
    target_fps=4
)

print_comparison(comparison)

Shape Comparison for A2020-EM-01-0184

SEQUENCE LENGTH:
  Expected: 141,852
  Actual:   141,852
  Match:    True
  Difference: 0

FEATURE COUNT:
  Expected: 30
  Actual:   30

✓ All shapes match perfectly!


## 6. Frame Interval의 영향 분석

FPS에 따라 frame_interval이 어떻게 변하는지 확인합니다.

In [ ]:
# 다양한 FPS 시나리오 테스트
def analyze_frame_interval_impact(record_id='A-0038'):
    """FPS 값에 따른 frame_interval 영향 분석"""
    
    result = track_video_shapes(record_id, verbose=False)
    
    if not result.video_exists:
        print(f"Video not found for {record_id}")
        return
    
    original_fps = result.fps
    frame_count = result.frame_count
    target_fps = 4
    
    print(f"\n{'='*80}")
    print(f"Frame Interval Impact Analysis for {record_id}")
    print(f"{'='*80}")
    print(f"Original FPS: {original_fps:.2f}")
    print(f"Frame Count: {frame_count:,}")
    print(f"Target FPS: {target_fps}")
    print(f"\n")
    
    # Calculate for different scenarios
    print(f"{'FPS':<10} {'Frame Interval':<20} {'Total Steps':<15} {'Sequence Length':<20}")
    print("-" * 70)
    
    for fps in [24, 25, 29.97, 30, 59.94, 60]:
        frame_interval = int(round(fps / target_fps))
        total_steps = max(1, int(frame_count // frame_interval) - 1)
        sequence_length = total_steps  # equals frames_processed
        
        print(f"{fps:<10.2f} {frame_interval:<20} {total_steps:<15,} {sequence_length:<20,}")
    
    print(f"\n" + "="*80)
    
    # Current video's calculation
    print(f"\nFor this specific video:")
    print(f"  FPS: {original_fps:.2f}")
    print(f"  Frame Interval: {result.frame_interval}")
    print(f"  Total Steps: {result.total_steps:,}")
    print(f"  Sequence Length: {result.expected_sequence_length:,}")
    print(f"\n" + "="*80)

analyze_frame_interval_impact('A-0038')

## 7. Total Steps 계산 상세 분석

`total_steps = max(1, int(frame_count // frame_interval) - 1)` 공식을 자세히 분석합니다.

In [ ]:
def analyze_total_steps_formula(record_id='A-0038'):
    """Total steps 계산 공식 상세 분석"""
    
    result = track_video_shapes(record_id, verbose=False)
    
    if not result.video_exists:
        print(f"Video not found for {record_id}")
        return
    
    frame_count = result.frame_count
    frame_interval = result.frame_interval
    
    print(f"\n{'='*80}")
    print(f"Total Steps Formula Analysis for {record_id}")
    print(f"{'='*80}\n")
    
    print(f"Given:")
    print(f"  frame_count = {frame_count:,}")
    print(f"  frame_interval = {frame_interval}")
    
    print(f"\nFormula: total_steps = max(1, int(frame_count // frame_interval) - 1)")
    
    print(f"\nStep-by-step calculation:")
    step1 = frame_count // frame_interval
    print(f"  1. frame_count // frame_interval = {frame_count:,} // {frame_interval} = {step1:,}")
    
    step2 = int(step1)
    print(f"  2. int({step1:,}) = {step2:,}")
    
    step3 = step2 - 1
    print(f"  3. {step2:,} - 1 = {step3:,}")
    
    step4 = max(1, step3)
    print(f"  4. max(1, {step3:,}) = {step4:,}")
    
    print(f"\nResult: total_steps = {step4:,}")
    
    print(f"\nWhy subtract 1?")
    print(f"  - The loop computes optical flow between consecutive frames")
    print(f"  - We need a previous frame and a current frame")
    print(f"  - First frame is used as 'previous', so we can't compute flow for it")
    print(f"  - Therefore, actual processing steps = (number of sampled frames) - 1")
    
    print(f"\nVerification:")
    print(f"  - Number of frames we can sample: {frame_count} // {frame_interval} = {frame_count // frame_interval:,}")
    print(f"  - Number of optical flow computations: {frame_count // frame_interval:,} - 1 = {(frame_count // frame_interval) - 1:,}")
    print(f"  - This matches our total_steps: {result.total_steps:,}")
    
    print(f"\n{'='*80}\n")

analyze_total_steps_formula('A-0038')

## 8. 시각화: Sequence Length vs Expected Epochs

여러 record의 sequence length와 expected epochs 관계를 시각화합니다.

In [ ]:
# 더 많은 record로 테스트
from pathlib import Path

# 사용 가능한 모든 record 찾기
video_base_path = "/tf/00_data/#_2021_Sleep_Video/"
all_records = [p.name for p in sorted(Path(video_base_path).glob("A-*")) if p.is_dir()]

print(f"Found {len(all_records)} records")
print(f"Testing first 20 records...\n")

test_records = all_records[:20]
results = batch_track_shapes(test_records, verbose=False)

# 데이터 추출
sequence_lengths = []
expected_epochs = []
ratios = []
record_names = []

for record_id, result in results.items():
    if result.video_exists and result.expected_epochs > 0:
        sequence_lengths.append(result.expected_sequence_length)
        expected_epochs.append(result.expected_epochs)
        ratios.append(result.expected_sequence_length / result.expected_epochs)
        record_names.append(record_id)

# 시각화
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Sequence Length vs Expected Epochs
axes[0, 0].scatter(expected_epochs, sequence_lengths, alpha=0.6)
axes[0, 0].set_xlabel('Expected Epochs (30s)')
axes[0, 0].set_ylabel('Sequence Length')
axes[0, 0].set_title('Sequence Length vs Expected Epochs')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Ratio distribution
axes[0, 1].hist(ratios, bins=20, edgecolor='black')
axes[0, 1].axvline(x=4*30, color='r', linestyle='--', label='Ideal (target_fps * 30 = 120)')
axes[0, 1].set_xlabel('Sequence Length / Expected Epochs')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Ratio Distribution')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Sequence length by record
axes[1, 0].bar(range(len(record_names)), sequence_lengths)
axes[1, 0].set_xlabel('Record Index')
axes[1, 0].set_ylabel('Sequence Length')
axes[1, 0].set_title('Sequence Length by Record')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Ratio by record
axes[1, 1].bar(range(len(record_names)), ratios)
axes[1, 1].axhline(y=4*30, color='r', linestyle='--', label='Ideal (120)')
axes[1, 1].set_xlabel('Record Index')
axes[1, 1].set_ylabel('Length / Epochs Ratio')
axes[1, 1].set_title('Ratio by Record')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 통계
print(f"\nStatistics:")
print(f"  Mean ratio: {np.mean(ratios):.4f}")
print(f"  Std ratio: {np.std(ratios):.4f}")
print(f"  Min ratio: {np.min(ratios):.4f}")
print(f"  Max ratio: {np.max(ratios):.4f}")
print(f"  Ideal ratio (target_fps * 30): {4 * 30}")

## 9. 특정 Record의 전체 Shape 흐름도

하나의 record에 대해 처음부터 끝까지의 shape 변화를 시각화합니다.

In [ ]:
def visualize_shape_flow(record_id='A-0038'):
    """Shape 변화 흐름도 출력"""
    
    result = track_video_shapes(record_id, verbose=False)
    
    if not result.video_exists:
        print(f"Video not found for {record_id}")
        return
    
    print(f"\n{'='*80}")
    print(f"Shape Flow Diagram for {record_id}")
    print(f"{'='*80}\n")
    
    print("INPUT: Video File")
    print(f"  └─ shape: {result.frame_count} frames × {result.frame_width} × {result.frame_height} pixels")
    print(f"  └─ fps: {result.fps:.2f}")
    print("")
    print("    ↓ (frame sampling with interval={})\n".format(result.frame_interval))
    print("")
    print("SAMPLED FRAMES")
    print(f"  └─ count: {result.total_steps + 1} frames (every {result.frame_interval}th frame)")
    print("")
    print("    ↓ (optical flow computation between consecutive frames)\n")
    print("")
    print("OPTICAL FLOW RESULTS")
    print(f"  └─ count: {result.total_steps} flows")
    print("")
    print("    ↓ (extract region magnitudes for each flow)\n")
    print("")
    print("REGION SEQUENCES (per region: Head, Body, Outer)")
    print(f"  ├─ v_seq: list of length {result.expected_sequence_length}")
    print(f"  └─ s_seq: list of length {result.expected_sequence_length}")
    print("")
    print("    ↓ (convert to numpy arrays)\n")
    print("")
    print("NUMPY ARRAYS (per region)")
    print(f"  ├─ v_seq: shape ({result.expected_sequence_length},)")
    print(f"  └─ s_seq: shape ({result.expected_sequence_length},)")
    print("")
    print("    ↓ (compute motion features)\n")
    print("")
    print("MOTION FEATURES (per region)")
    print(f"  ├─ f1@30s:  shape ({result.expected_sequence_length},)")
    print(f"  ├─ f1@300s: shape ({result.expected_sequence_length},)")
    print(f"  ├─ f2@30s:  shape ({result.expected_sequence_length},)")
    print(f"  ├─ f2@300s: shape ({result.expected_sequence_length},)")
    print(f"  ├─ f3@0.01: shape ({result.expected_sequence_length},)")
    print(f"  ├─ f3@0.1:  shape ({result.expected_sequence_length},)")
    print(f"  ├─ f3@1.0:  shape ({result.expected_sequence_length},)")
    print(f"  ├─ f4@0.01: shape ({result.expected_sequence_length},)")
    print(f"  ├─ f4@0.1:  shape ({result.expected_sequence_length},)")
    print(f"  └─ f4@1.0:  shape ({result.expected_sequence_length},)")
    print("")
    print("    ↓ (combine all regions)\n")
    print("")
    print("FINAL OUTPUT")
    print(f"  └─ Dictionary with {len(result.feature_shapes)} features")
    print(f"     Each feature: shape ({result.expected_sequence_length},)")
    print("")
    print(f"{'='*80}\n")
    
    print(f"SUMMARY:")
    print(f"  Video frames: {result.frame_count:,} → Feature length: {result.expected_sequence_length:,}")
    print(f"  Compression ratio: {result.frame_count / result.expected_sequence_length:.2f}x")
    print(f"  Expected epochs: {result.expected_epochs:.2f}")
    print(f"  Feature length / epochs: {result.expected_sequence_length / result.expected_epochs:.4f}")
    print(f"\n{'='*80}\n")

visualize_shape_flow('A-0038')

## 10. Export Results

분석 결과를 CSV로 저장합니다.

In [ ]:
# 모든 테스트 record의 shape 정보를 CSV로 저장
if 'results' in locals() and results:
    export_data = []
    for record_id, result in results.items():
        if result.video_exists:
            export_data.append({
                'record_id': record_id,
                'fps': result.fps,
                'frame_count': result.frame_count,
                'frame_width': result.frame_width,
                'frame_height': result.frame_height,
                'duration_seconds': result.duration_seconds,
                'target_fps': result.target_fps,
                'frame_interval': result.frame_interval,
                'total_steps': result.total_steps,
                'sequence_length': result.expected_sequence_length,
                'expected_epochs': result.expected_epochs,
                'length_to_epochs_ratio': result.expected_sequence_length / result.expected_epochs if result.expected_epochs > 0 else 0,
                'feature_count': len(result.feature_shapes),
            })
    
    df_export = pd.DataFrame(export_data)
    output_file = 'motion_shape_tracking_results.csv'
    df_export.to_csv(output_file, index=False)
    print(f"Results exported to: {output_file}")
    print(f"\nPreview:")
    print(df_export.head(10).to_string(index=False))
else:
    print("No results to export. Run the batch tracking cell first.")